# Convergence Study — ±2 Yard Aim Tolerance

This notebook repeats every analysis from `convergence_analysis.ipynb` but uses **±2 yards** as the aim-offset tolerance (= one full grid step, the smallest meaningful difference in the aim grid).

It also adds:
- Separate heatmaps for **club only**, **aim only**, and **joint (club + aim)** convergence
- All convergence metrics **overlaid directly on the hole layout** so you can see exactly which positions on the fairway converge and what strategy is chosen there

## 0. Setup

In [15]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns

HERE    = Path(".").resolve()    # absolute path to results/
PARENT  = HERE.parent            # absolute path to convergence/
PLOTS   = HERE / "plots"
PLOTS.mkdir(exist_ok=True)

# Add parent to path so we can import core.py
sys.path.insert(0, str(PARENT))
matplotlib.use("Agg")   # must be set before core imports it
from core import build_hole, plot_hole_layout, CLUB_STYLES

# ── Constants ────────────────────────────────────────────────────────────────
AIM_TOL     = 2.0        # yards — the tolerance we use throughout this notebook
CONV_THRESH = 0.80       # fraction of seeds that must agree to call a point "converged"

DISTANCE_BANDS = [(50, 80), (80, 120), (120, 180), (180, 250), (250, 320)]
BAND_LABELS    = ["50-80 yd", "80-120 yd", "120-180 yd", "180-250 yd", "250+ yd"]

def band_label(y):
    for (lo, hi), lbl in zip(DISTANCE_BANDS, BAND_LABELS):
        if lo <= y < hi:
            return lbl
    return "250+ yd"

cmap_rg = plt.get_cmap("RdYlGn")
norm01   = mcolors.Normalize(vmin=0, vmax=1)

print(f"HERE   = {HERE}")
print(f"PARENT = {PARENT}")
print("Imports OK")

HERE   = /Users/federicadomecq/gitrepos/golfOnPar/Parallelisation/convergence/results
PARENT = /Users/federicadomecq/gitrepos/golfOnPar/Parallelisation/convergence
Imports OK


In [16]:
# ── Load pre-computed strategy data ──────────────────────────────────────────
strategies = pd.read_csv(HERE / "strategies_key_N.csv")
match_rates_orig = pd.read_csv(HERE / "match_rates_all_seeds.csv")
match_rates_orig["match_rate_pct"] = pd.to_numeric(
    match_rates_orig["match_rate_pct"], errors="coerce"
)

N_VALS = sorted(strategies["N"].unique())
SEEDS  = sorted(strategies["seed"].unique())
print(f"Strategy data: {len(strategies):,} rows | seeds={len(SEEDS)} | N={N_VALS}")

Strategy data: 252,000 rows | seeds=100 | N=[10, 20, 30, 50, 100, 150, 200, 250, 300]


In [17]:
# ── Build hole geometry (needed for all overlay plots) ───────────────────────
# This trains the putting GP — takes ~30 seconds.
DATA_DIR = PARENT.parent / "data"
hole = build_hole(DATA_DIR, gp_training_iter=100)
print(f"Hole built. Pin at {hole.hole}, {len(hole.strategy_points)} strategy points.")

Hole built. Pin at (5, 334), 280 strategy points.


---
## 1. Compute Agreement Metrics (±2 yd)

For each `(x, y, N)` group across all 100 seeds we compute three agreement fractions:

| Metric | Definition |
|--------|-----------|
| **club_agreement** | Fraction of seeds that chose the **modal club** |
| **aim_agreement** | Fraction of seeds whose aim offset is **within ±2 yd of the cross-seed median** |
| **joint_agreement** | Fraction of seeds where club = modal club **AND** aim is within ±2 yd of median aim |

We also record the consensus values (majority club, median aim) which appear in the overlay plots.

In [18]:
records = []
for (x, y, n), grp in strategies.groupby(["x", "y", "N"]):
    mode_club    = grp["club"].mode().iloc[0]
    median_aim   = grp["aim_offset"].median()
    club_agree   = (grp["club"] == mode_club).mean()
    aim_agree    = (grp["aim_offset"] - median_aim).abs().le(AIM_TOL).mean()
    joint_agree  = ((grp["club"] == mode_club) &
                    (grp["aim_offset"] - median_aim).abs().le(AIM_TOL)).mean()
    records.append(dict(
        x=x, y=y, N=n,
        club_agreement=club_agree,
        aim_agreement=aim_agree,
        joint_agreement=joint_agree,
        majority_club=mode_club,
        median_aim=median_aim,
        club_short=CLUB_STYLES.get(mode_club, {}).get("short", mode_club),
        club_color=CLUB_STYLES.get(mode_club, {}).get("color", "#999999"),
        n_seeds=len(grp),
    ))

ag = pd.DataFrame(records)
ag["band"] = ag["y"].apply(band_label)

at300 = ag[ag["N"] == 300].copy()
print(f"Agreement table: {len(ag):,} rows")
print(f"\nAt N=300  (AIM_TOL=±{AIM_TOL} yd):")
print(f"  Club agreement   — mean {at300['club_agreement'].mean():.1%}  |  "
      f"points ≥80%: {(at300['club_agreement']>=CONV_THRESH).sum()}/{len(at300)}")
print(f"  Aim  agreement   — mean {at300['aim_agreement'].mean():.1%}  |  "
      f"points ≥80%: {(at300['aim_agreement']>=CONV_THRESH).sum()}/{len(at300)}")
print(f"  Joint agreement  — mean {at300['joint_agreement'].mean():.1%}  |  "
      f"points ≥80%: {(at300['joint_agreement']>=CONV_THRESH).sum()}/{len(at300)}")

Agreement table: 2,520 rows

At N=300  (AIM_TOL=±2.0 yd):
  Club agreement   — mean 93.8%  |  points ≥80%: 245/280
  Aim  agreement   — mean 77.1%  |  points ≥80%: 159/280
  Joint agreement  — mean 73.9%  |  points ≥80%: 141/280


In [19]:
# ── Sequential match rate with ±2 yd tolerance ───────────────────────────────
n_pairs = list(zip(N_VALS[:-1], N_VALS[1:]))
mr_records = []
for seed in SEEDS:
    sdf = strategies[strategies["seed"] == seed].set_index(["x", "y"])
    for n_from, n_to in n_pairs:
        prev = sdf[sdf["N"] == n_from][["club", "aim_offset"]]
        curr = sdf[sdf["N"] == n_to  ][["club", "aim_offset"]]
        both = prev.join(curr, lsuffix="_p", rsuffix="_c", how="inner")
        if both.empty:
            continue
        same_club = both["club_p"] == both["club_c"]
        same_aim  = (both["aim_offset_p"] - both["aim_offset_c"]).abs() <= AIM_TOL
        mr_records.append(dict(
            seed=seed, N=n_to,
            match_rate=(same_club & same_aim).mean() * 100,
        ))

mr2 = pd.DataFrame(mr_records)
print("Sequential match rate (±2 yd) computed.")
print(mr2.groupby("N")["match_rate"].mean().round(1).to_string())

Sequential match rate (±2 yd) computed.
N
20     46.6
30     60.7
50     61.6
100    61.6
150    73.1
200    78.6
250    82.3
300    84.9


---
## 2. Match Rate Curves — ±1 yd vs ±2 yd

Showing the original ±1 yd match rate (from the simulation logs) alongside the ±2 yd recomputed version, so the effect of relaxing the tolerance is visible directly.

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, label, color) in zip(
    axes,
    [
        (match_rates_orig, "±1 yd (original)", "#E91E63"),
        (mr2,              "±2 yd",            "#4CAF50"),
    ]
):
    col = "match_rate_pct" if "match_rate_pct" in df.columns else "match_rate"
    piv = df.pivot_table(index="N", columns="seed", values=col, aggfunc="mean")
    mu  = piv.mean(axis=1)
    std = piv.std(axis=1)
    ax.plot(piv.index, piv.values, color=color, alpha=0.08, linewidth=0.7)
    ax.fill_between(piv.index, mu-std, mu+std, alpha=0.20, color=color)
    ax.plot(piv.index, mu, color=color, linewidth=2.5, label=f"Mean  {label}")
    ax.axhline(80, color="grey", linestyle="--", linewidth=1, label="80% threshold")
    final = piv.iloc[-1].dropna()
    ax.set_title(f"Match Rate Convergence  ({label})\nFinal mean={final.mean():.1f}%")
    ax.set_xlabel("N (shots per grid point)")
    ax.set_ylabel("Match rate vs previous snapshot (%)")
    ax.set_ylim(0, 105); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS / "2yd_match_rate_curves.png", dpi=150)
plt.show()

/var/folders/vr/34gvhdpn06z_plwsw13j8w4c0000gn/T/ipykernel_70074/4010821784.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Three-Way Heatmaps — Club / Aim / Joint at N=300

Each cell is one `(x, y)` grid point. Rows = distance from tee (y), columns = lateral position (x).  
Colour = fraction of seeds that agreed under each metric.

- **Club only** — did seeds agree on which club to use?
- **Aim only (±2 yd)** — did seeds aim within 2 yards of each other?
- **Joint (±2 yd)** — did seeds agree on *both*?

In [21]:
metrics = [
    ("club_agreement",  "Club Agreement"),
    ("aim_agreement",   f"Aim Agreement  (±{AIM_TOL} yd)"),
    ("joint_agreement", f"Joint  Club + Aim  (±{AIM_TOL} yd)"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True)
for ax, (col, title) in zip(axes, metrics):
    piv = at300.pivot_table(index="y", columns="x", values=col, aggfunc="mean")
    piv = piv.sort_index(ascending=False)
    im  = ax.imshow(piv.values, aspect="auto", vmin=0, vmax=1,
                    cmap="RdYlGn", origin="upper")
    plt.colorbar(im, ax=ax, label="Fraction of seeds agreeing", shrink=0.7)
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f"{v:.0f}" for v in piv.columns],
                       rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels([f"{v:.0f}" for v in piv.index], fontsize=7)
    ax.set_xlabel("x  (lateral yards)")
    ax.set_title(title, fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel("y  (yards from tee)")

plt.suptitle(f"Cross-Seed Agreement at N=300  (aim tolerance = ±{AIM_TOL} yd)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS / "2yd_heatmaps_N300.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/vr/34gvhdpn06z_plwsw13j8w4c0000gn/T/ipykernel_70074/1438225518.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Hole Overlay — Agreement Levels

Now we draw the same three metrics **directly on the hole geometry**.  
Each dot = one strategy grid point. Colour = agreement fraction (red → yellow → green).  
The hole, fairway, water, and bunker outlines give spatial context.

In [22]:
matplotlib.use("Agg")  # keep consistent with core.py
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

cmap_rg = plt.get_cmap("RdYlGn")
norm01   = mcolors.Normalize(vmin=0, vmax=1)

fig, axes = plt.subplots(1, 3, figsize=(22, 14))
titles = ["Club Agreement", f"Aim Agreement (±{AIM_TOL} yd)", f"Joint Agreement (±{AIM_TOL} yd)"]
cols   = ["club_agreement", "aim_agreement", "joint_agreement"]

for ax, col, title in zip(axes, cols, titles):
    plot_hole_layout(hole, title=title, plot_strategy_points=False, ax=ax)
    sc = ax.scatter(
        at300["x"], at300["y"],
        c=at300[col], cmap="RdYlGn", vmin=0, vmax=1,
        s=120, edgecolors="k", linewidths=0.4, zorder=20, alpha=0.92,
    )
    plt.colorbar(sc, ax=ax, label="Agreement fraction", shrink=0.55, pad=0.02)
    ax.set_xlim(-55, 75); ax.set_ylim(40, 300)

plt.suptitle(f"Cross-Seed Agreement at N=300 on Hole Layout  (aim tol=±{AIM_TOL} yd)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS / "2yd_hole_agreement_overlay.png", dpi=130, bbox_inches="tight")
plt.show()

/var/folders/vr/34gvhdpn06z_plwsw13j8w4c0000gn/T/ipykernel_70074/244558965.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Hole Overlay — Consensus Strategy with Convergence Labels

This is the main diagnostic plot.

**Each dot shows what the strategy actually *is* at that point:**
- **Dot colour** = convergence level (green = converged, red = not converged), using **club agreement**
- **Dot size** = scales with joint agreement — larger dots are more confident decisions
- **Text label** = `Club, aim` e.g. `Hy,+4` — the majority club and median aim offset
  - Labels shown in **black** for converged points (club agreement ≥ 80 %)
  - Labels shown in **red** for non-converged points
- **Aim arrow** = a short arrow from each point toward the aim direction (pin ± aim offset), showing where the majority strategy is actually aiming

In [23]:
fig, ax = plt.subplots(figsize=(14, 18))
plot_hole_layout(hole, title="Consensus Strategy at N=300  (colour = club convergence)",
                 plot_strategy_points=False, ax=ax)

pin_x, pin_y = hole.hole

for _, row in at300.iterrows():
    px, py     = row["x"], row["y"]
    club_ag    = row["club_agreement"]
    joint_ag   = row["joint_agreement"]
    club_short = row["club_short"]
    aim        = row["median_aim"]

    # Dot: coloured by club agreement, sized by joint agreement
    dot_color = cmap_rg(norm01(club_ag))
    dot_size  = 60 + 220 * joint_ag   # 60 (no agreement) → 280 (full agreement)
    ax.scatter(px, py, c=[dot_color], s=dot_size,
               edgecolors="k", linewidths=0.5, zorder=20, alpha=0.88)

    # Aim arrow: from grid point toward (pin_x + aim_offset, pin_y)
    aim_target_x = pin_x + aim
    direction    = np.array([aim_target_x - px, pin_y - py])
    dist         = np.linalg.norm(direction)
    if dist > 0:
        arrow_len = min(12, dist * 0.18)   # cap at 12 yards so arrows don't overlap
        unit      = direction / dist
        ax.annotate(
            "", xy=(px + unit[0]*arrow_len, py + unit[1]*arrow_len),
            xytext=(px, py),
            arrowprops=dict(arrowstyle="-|>", color="dimgrey",
                            lw=0.8, mutation_scale=7),
            zorder=21,
        )

    # Text label: "Club,+aim"
    label      = f"{club_short},{int(aim):+}"
    label_color = "black" if club_ag >= CONV_THRESH else "red"
    ax.text(px + 1.5, py + 3, label, fontsize=5.5,
            color=label_color, zorder=22, fontweight="bold" if club_ag >= CONV_THRESH else "normal")

# Colourbar for agreement
sm = cm.ScalarMappable(cmap="RdYlGn", norm=norm01)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label("Club agreement fraction  (green = converged)")

# Legend: dot size
for ag_val, label in [(0.5, "50% joint"), (0.8, "80% joint"), (1.0, "100% joint")]:
    ax.scatter([], [], s=60+220*ag_val, c="grey", edgecolors="k", alpha=0.7, label=label)
ax.legend(title="Joint agreement", loc="upper left", fontsize=8, title_fontsize=8)

ax.set_xlim(-55, 75); ax.set_ylim(40, 300)
plt.tight_layout()
plt.savefig(PLOTS / "2yd_hole_strategy_overlay.png", dpi=130, bbox_inches="tight")
plt.show()

/var/folders/vr/34gvhdpn06z_plwsw13j8w4c0000gn/T/ipykernel_70074/2387124449.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Hole Overlay — Strategy Evolution Across N

Five snapshots (N = 50, 100, 150, 200, 300) on the same hole layout, so you can watch the strategy stabilise spatially. Labels and colours are the same as above.

This lets you see which parts of the hole *settle first* and which stay uncertain for longer.

In [24]:
snap_N = [n for n in [50, 100, 150, 200, 300] if n in N_VALS]
fig, axes = plt.subplots(1, len(snap_N), figsize=(6*len(snap_N), 16), sharey=True)

for ax, n_val in zip(axes, snap_N):
    snap = ag[ag["N"] == n_val].copy()
    plot_hole_layout(hole, title=f"N = {n_val}",
                     plot_strategy_points=False, ax=ax)

    for _, row in snap.iterrows():
        px, py  = row["x"], row["y"]
        ag_club = row["club_agreement"]
        ag_jt   = row["joint_agreement"]

        dot_color = cmap_rg(norm01(ag_club))
        dot_size  = 50 + 200 * ag_jt
        ax.scatter(px, py, c=[dot_color], s=dot_size,
                   edgecolors="k", linewidths=0.4, zorder=20, alpha=0.88)

        # Only annotate converged points to keep plot readable
        if ag_club >= CONV_THRESH:
            label = f"{row['club_short']},{int(row['median_aim']):+}"
            ax.text(px + 1.5, py + 3, label, fontsize=4.5,
                    color="black", zorder=22)

    ax.set_xlim(-55, 75); ax.set_ylim(40, 300)
    # small colourbar per panel
    sm = cm.ScalarMappable(cmap="RdYlGn", norm=norm01)
    sm.set_array([])
    pct = (snap["club_agreement"] >= CONV_THRESH).mean() * 100
    ax.set_xlabel(f"x (yards)\n{pct:.0f}% of points converged")

plt.suptitle("Strategy Convergence Over N  (colour = club agreement, size = joint agreement)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS / "2yd_hole_evolution.png", dpi=130, bbox_inches="tight")
plt.show()

/var/folders/vr/34gvhdpn06z_plwsw13j8w4c0000gn/T/ipykernel_70074/1518277748.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7. Hole Overlay — Club Identity Map

Separate from convergence: what club does the **consensus strategy** prescribe at each grid point?  
Dot colour = majority club (using the same colour scheme as the simulation PNGs).  
Dot size = club agreement (bigger = more confident).  
Label = club short name + median aim offset.

This is the final strategy the model has converged to.

In [25]:
fig, ax = plt.subplots(figsize=(14, 18))
plot_hole_layout(hole, title="Consensus Club Strategy at N=300",
                 plot_strategy_points=False, ax=ax)

for _, row in at300.iterrows():
    px, py     = row["x"], row["y"]
    club_ag    = row["club_agreement"]
    dot_color  = row["club_color"]
    # Edge: black=converged, red=ambiguous
    edge_color = "black" if club_ag >= CONV_THRESH else "red"
    edge_w     = 0.5      if club_ag >= CONV_THRESH else 1.8
    dot_size   = 40 + 240 * club_ag

    ax.scatter(px, py, c=[dot_color], s=dot_size,
               edgecolors=edge_color, linewidths=edge_w,
               zorder=20, alpha=0.90)

    aim      = row["median_aim"]
    label    = f"{row['club_short']},{int(aim):+}"
    ax.text(px + 1.5, py + 3, label, fontsize=5.5, zorder=22,
            color="black" if club_ag >= CONV_THRESH else "crimson",
            fontweight="bold")

# Club colour legend
seen_clubs = at300["majority_club"].unique()
patches = [
    mpatches.Patch(
        facecolor=CLUB_STYLES.get(c, {}).get("color", "#999"),
        edgecolor="k", linewidth=0.5,
        label=f"{CLUB_STYLES.get(c,{}).get('short',c)}  ({c})"
    )
    for c in sorted(seen_clubs)
    if c in CLUB_STYLES
]
ax.legend(handles=patches, title="Majority Club", loc="upper left",
          fontsize=7, title_fontsize=8, ncol=2)

ax.set_xlim(-55, 75); ax.set_ylim(40, 300)
ax.text(0.99, 0.01, "Red border = club agreement < 80%",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color="crimson")

plt.tight_layout()
plt.savefig(PLOTS / "2yd_hole_club_identity.png", dpi=130, bbox_inches="tight")
plt.show()

/var/folders/vr/34gvhdpn06z_plwsw13j8w4c0000gn/T/ipykernel_70074/3786284879.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8. Distance Band Analysis (±2 yd)

Replicating the distance-band convergence curves with the ±2 yd tolerance.

In [26]:
palette = dict(zip(BAND_LABELS, sns.color_palette("viridis", len(BAND_LABELS))))
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

# Panel A: mean aim/club/joint agreement vs N per band
for ax, col, title in zip(
    axes,
    ["club_agreement", "aim_agreement", "joint_agreement"],
    ["Club Agreement", f"Aim Agreement (±{AIM_TOL} yd)", f"Joint Agreement (±{AIM_TOL} yd)"],
):
    grp = (ag.groupby(["N", "band"])[col]
             .agg(mean="mean", std="std")
             .reset_index())
    for band in BAND_LABELS:
        sub = grp[grp["band"] == band].sort_values("N")
        if sub.empty:
            continue
        ax.plot(sub["N"], sub["mean"] * 100, label=band,
                color=palette[band], linewidth=2, marker="o", markersize=3)
        ax.fill_between(sub["N"],
                        (sub["mean"] - sub["std"]) * 100,
                        (sub["mean"] + sub["std"]) * 100,
                        alpha=0.10, color=palette[band])
    ax.axhline(80, color="grey", linestyle="--", linewidth=0.9)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("N"); ax.set_ylabel("Mean agreement (%)"); ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8, loc="lower right")
plt.suptitle("Agreement by Distance Band  (±2 yd aim tolerance)", fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS / "2yd_agreement_by_distance.png", dpi=150)
plt.show()

/var/folders/vr/34gvhdpn06z_plwsw13j8w4c0000gn/T/ipykernel_70074/1316037802.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 9. Summary Table & Minimum-N Recommendation

In [27]:
# Per-N summary: match rate + fraction of points converged under each metric
rows = []
for n in N_VALS:
    mr_vals   = mr2[mr2["N"] == n]["match_rate"].dropna()
    snap      = ag[ag["N"] == n]
    rows.append({
        "N": n,
        "Match rate mean": f"{mr_vals.mean():.1f}%",
        "Match rate p10":  f"{mr_vals.quantile(0.1):.1f}%",
        "Club ≥80% pts":   f"{(snap['club_agreement']  >=CONV_THRESH).mean()*100:.0f}%",
        "Aim ≥80% pts":    f"{(snap['aim_agreement']   >=CONV_THRESH).mean()*100:.0f}%",
        "Joint ≥80% pts":  f"{(snap['joint_agreement'] >=CONV_THRESH).mean()*100:.0f}%",
    })
display(pd.DataFrame(rows).set_index("N"))

,Match rate mean,Match rate p10,Club ≥80% pts,Aim ≥80% pts,Joint ≥80% pts
N,,,,,
10,nan%,nan%,53%,5%,4%
20,46.6%,43.2%,65%,9%,8%
30,60.7%,56.8%,70%,12%,9%
50,61.6%,58.2%,74%,18%,14%
100,61.6%,58.2%,80%,28%,24%
150,73.1%,70.4%,84%,39%,34%
200,78.6%,75.0%,84%,47%,41%
250,82.3%,80.0%,86%,52%,44%
300,84.9%,82.5%,88%,57%,50%


In [28]:
print(f"""
AIM TOLERANCE = ±{AIM_TOL} yd  |  CONVERGENCE THRESHOLD = {CONV_THRESH:.0%}
══════════════════════════════════════════════════════════

SEQUENTIAL MATCH RATE (±{AIM_TOL} yd)
  N=100 : {mr2[mr2.N==100]['match_rate'].mean():.1f}%  (p10={mr2[mr2.N==100]['match_rate'].quantile(.1):.1f}%)
  N=150 : {mr2[mr2.N==150]['match_rate'].mean():.1f}%
  N=200 : {mr2[mr2.N==200]['match_rate'].mean():.1f}%
  N=250 : {mr2[mr2.N==250]['match_rate'].mean():.1f}%
  N=300 : {mr2[mr2.N==300]['match_rate'].mean():.1f}%

GRID-POINT CONVERGENCE AT N=300
  Club   ≥80%: {(at300['club_agreement']  >=CONV_THRESH).mean()*100:.0f}% of points  (mean agree {at300['club_agreement'].mean()*100:.1f}%)
  Aim    ≥80%: {(at300['aim_agreement']   >=CONV_THRESH).mean()*100:.0f}% of points  (mean agree {at300['aim_agreement'].mean()*100:.1f}%)
  Joint  ≥80%: {(at300['joint_agreement'] >=CONV_THRESH).mean()*100:.0f}% of points  (mean agree {at300['joint_agreement'].mean()*100:.1f}%)

BY DISTANCE BAND — club agreement at N=300:
""")
for band in BAND_LABELS:
    sub = at300[at300["band"] == band]
    nc  = (sub["club_agreement"] < CONV_THRESH).sum()
    print(f"  {band:12s}: mean={sub['club_agreement'].mean()*100:.1f}%  "
          f"not-converged={nc}/{len(sub)}")


AIM TOLERANCE = ±2.0 yd  |  CONVERGENCE THRESHOLD = 80%
══════════════════════════════════════════════════════════

SEQUENTIAL MATCH RATE (±2.0 yd)
  N=100 : 61.6%  (p10=58.2%)
  N=150 : 73.1%
  N=200 : 78.6%
  N=250 : 82.3%
  N=300 : 84.9%

GRID-POINT CONVERGENCE AT N=300
  Club   ≥80%: 88% of points  (mean agree 93.8%)
  Aim    ≥80%: 57% of points  (mean agree 77.1%)
  Joint  ≥80%: 50% of points  (mean agree 73.9%)

BY DISTANCE BAND — club agreement at N=300:

  50-80 yd    : mean=97.7%  not-converged=1/40
  80-120 yd   : mean=83.2%  not-converged=17/50
  120-180 yd  : mean=96.0%  not-converged=5/60
  180-250 yd  : mean=94.8%  not-converged=10/90
  250+ yd     : mean=97.5%  not-converged=2/40
